<style>
.rendered_html h1 { color: #1f4e79; }
.rendered_html h2 { color: #0f766e; }
.rendered_html h3, .rendered_html h4 { color: #7c3aed; }
</style>

# Evaluation Metrics for Implicit Recommendation

This notebook evaluates the baseline SGD Matrix Factorization model saved from notebook 02.

## Setup

We load the processed data and saved model. Evaluation uses the trained model; we do not retrain here. This keeps the workflow organized.

In [1]:
from pathlib import Path
import math
import pickle

import numpy as np
import pandas as pd
import torch
from torch import nn

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

processed_dir = project_root / "data" / "processed"
models_dir = project_root / "models"
tables_dir = project_root / "results" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)

## Load Data and Model Checkpoint

The train/test split and mappings come from notebook 01. The model weights come from notebook 02.

In [2]:
train_data = pd.read_csv(processed_dir / "train_data.csv")
test_data = pd.read_csv(processed_dir / "test_data.csv")
movies = pd.read_csv(processed_dir / "movies_processed.csv")

with open(processed_dir / "metadata.pkl", "rb") as file:
    metadata = pickle.load(file)

model_path = models_dir / "mf_sgd.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

checkpoint = torch.load(model_path, map_location=device)

num_users = checkpoint.get("num_users", metadata["num_users"])
num_movies = checkpoint.get("num_movies", metadata["num_movies"])
embedding_dim = checkpoint.get("embedding_dim", 32)

print("Train data:", train_data.shape)
print("Test data:", test_data.shape)
print("Movies:", movies.shape)

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Train data: (54433, 7)
Test data: (942, 7)
Movies: (1682, 25)


## Recreate Model Class

The architecture must match notebook 02 so the saved weights load correctly.

In [3]:
class ImplicitMatrixFactorization(nn.Module):
    def __init__(self, num_users, num_movies, embedding_dim=32, use_bias=True):
        super().__init__()
        self.user_embeddings = nn.Embedding(num_users, embedding_dim)
        self.movie_embeddings = nn.Embedding(num_movies, embedding_dim)
        self.use_bias = use_bias

        if use_bias:
            self.user_bias = nn.Embedding(num_users, 1)
            self.movie_bias = nn.Embedding(num_movies, 1)

    def forward(self, user_idx, movie_idx):
        user_vector = self.user_embeddings(user_idx)
        movie_vector = self.movie_embeddings(movie_idx)
        score = (user_vector * movie_vector).sum(dim=1)

        if self.use_bias:
            score = score + self.user_bias(user_idx).squeeze(1)
            score = score + self.movie_bias(movie_idx).squeeze(1)

        return score


model = ImplicitMatrixFactorization(
    num_users=num_users,
    num_movies=num_movies,
    embedding_dim=embedding_dim,
    use_bias=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

print("Loaded model from:", model_path)

Loaded model from: c:\Users\abdoi\OneDrive\Documents\Studies\master ai\S2\Optimisation\projet\workspace\models\mf_sgd.pth


## Top-K Recommendation Function

We score all movies for a user, then remove movies already seen in training. Recommender systems should suggest new items; recommending already-seen movies is not useful.

In [4]:
K = 10

seen_movies_by_user = train_data.groupby("user_idx")["movie_idx"].apply(set).to_dict()
all_movie_indices = torch.arange(num_movies, dtype=torch.long, device=device)


def recommend_top_k(user_idx, k=10):
    with torch.no_grad():
        user_tensor = torch.full((num_movies,), int(user_idx), dtype=torch.long, device=device)
        scores = model(user_tensor, all_movie_indices).detach().cpu().numpy()

    seen_movies = seen_movies_by_user.get(user_idx, set())
    if seen_movies:
        scores[list(seen_movies)] = -np.inf

    top_k_movie_idx = np.argsort(scores)[-k:][::-1]
    return top_k_movie_idx.tolist()

## Metrics@K

HitRate@K checks whether the true test movie appears in the top K recommendations.

Precision@K asks: among the K recommended movies, how many are relevant?

Recall@K asks: among the relevant movies, how many were found?

NDCG@K rewards putting the relevant movie closer to the top of the list.

In [5]:
def hit_rate_at_k(recommended_items, test_item):
    return int(test_item in recommended_items)


def precision_at_k(recommended_items, test_item, k=10):
    return hit_rate_at_k(recommended_items, test_item) / k


def recall_at_k(recommended_items, test_item):
    return hit_rate_at_k(recommended_items, test_item)


def ndcg_at_k(recommended_items, test_item):
    if test_item not in recommended_items:
        return 0.0

    rank = recommended_items.index(test_item) + 1
    return 1 / math.log2(rank + 1)

## Evaluate All Test Users

Leave-one-out gives each user one true test movie. We average the metrics across all test users.

In [6]:
precision_scores = []
recall_scores = []
hit_rate_scores = []
ndcg_scores = []

for row in test_data.itertuples(index=False):
    recommendations = recommend_top_k(row.user_idx, k=K)
    test_movie_idx = int(row.movie_idx)

    precision_scores.append(precision_at_k(recommendations, test_movie_idx, k=K))
    recall_scores.append(recall_at_k(recommendations, test_movie_idx))
    hit_rate_scores.append(hit_rate_at_k(recommendations, test_movie_idx))
    ndcg_scores.append(ndcg_at_k(recommendations, test_movie_idx))

metrics = {
    "metric": ["Precision@10", "Recall@10", "HitRate@10", "NDCG@10"],
    "value": [
        np.mean(precision_scores),
        np.mean(recall_scores),
        np.mean(hit_rate_scores),
        np.mean(ndcg_scores),
    ]
}

metrics_df = pd.DataFrame(metrics)
display(metrics_df)

metrics_path = tables_dir / "sgd_evaluation_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

print("Saved metrics to:", metrics_path)

,metric,value
0,Precision@10,0.008068
1,Recall@10,0.080679
2,HitRate@10,0.080679
3,NDCG@10,0.040565


Saved metrics to: c:\Users\abdoi\OneDrive\Documents\Studies\master ai\S2\Optimisation\projet\workspace\results\tables\sgd_evaluation_metrics.csv


## Example User

We show one user's true test movie and the Top-10 recommended movie titles.

In [7]:
example = test_data.iloc[0]
example_user_idx = int(example["user_idx"])
true_movie_idx = int(example["movie_idx"])
true_movie_id = metadata["idx_to_movie"][true_movie_idx]

recommended_movie_indices = recommend_top_k(example_user_idx, k=K)
recommended_movie_ids = [metadata["idx_to_movie"][idx] for idx in recommended_movie_indices]
found_true_item = true_movie_idx in recommended_movie_indices

true_title = movies.loc[movies["movie_id"] == true_movie_id, "title"].iloc[0]

recommendations_df = pd.DataFrame({
    "rank": range(1, K + 1),
    "movie_idx": recommended_movie_indices,
    "movie_id": recommended_movie_ids,
}).merge(movies[["movie_id", "title"]], on="movie_id", how="left")

print("User idx:", example_user_idx)
print("True test movie:", true_title)
print("Found in Top-10:", found_true_item)

display(recommendations_df[["rank", "movie_idx", "title"]])

User idx: 0
True test movie: When the Cats Away (Chacun cherche son chat) (1996)
Found in Top-10: False


,rank,movie_idx,title
0,1,285,"English Patient, The (1996)"
1,2,312,Titanic (1997)
2,3,317,Schindler's List (1993)
3,4,299,Air Force One (1997)
4,5,287,Scream (1996)
5,6,116,"Rock, The (1996)"
6,7,356,One Flew Over the Cuckoo's Nest (1975)
7,8,236,Jerry Maguire (1996)
8,9,68,Forrest Gump (1994)
9,10,301,L.A. Confidential (1997)


## Conclusion

These metrics show how well the model ranks the user's held-out liked movie. Accuracy is not enough for recommendation because we care about ranking useful items near the top, not classifying every possible user-movie pair.

The next notebook will compare optimizers using the same evaluation method.